# 06 · Agentic AI: an analyst agent that writes and runs its own SQL

**Generative AI** answers once. An **agent** is given a goal and a **tool**, then loops on its own: *think → call the tool → read the result → fix mistakes → answer.*

Here the tool is `run_sql`: a **read-only** SQL runner with **guardrails**. We then **test the agent** on 10 business questions whose correct answers we compute ourselves.

Run cells one at a time, top to bottom.

In [1]:
# 1. Setup
import os, re, time, sqlite3
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
MODEL = "gemini-flash-lite-latest"
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"), http_options=types.HttpOptions(timeout=180000))
os.makedirs("outputs", exist_ok=True)

In [2]:
# 2. Build the complaints database (SQLite file) - narratives left out, the agent only needs the facts
cols = ["complaint_id", "date_received", "issue", "sub_issue", "company", "state", "tags",
        "company_response_to_consumer", "timely_response", "words"]
allc = pd.concat([pd.read_csv("data/clean/train.csv", usecols=cols),
                  pd.read_csv("data/clean/test_1000.csv", usecols=cols)], ignore_index=True)
d = pd.to_datetime(allc["date_received"], format="mixed", utc=True)
allc["year"] = d.dt.year
allc["month"] = d.dt.strftime("%Y-%m")
allc = allc.drop(columns="date_received").rename(columns={"company_response_to_consumer": "company_response"})

tri = pd.read_csv("outputs/llm_triage.csv").drop_duplicates("complaint_id")
tri["vulnerable"] = tri["vulnerable"].astype(str).str.lower().eq("true").astype(int)
tri = tri[["complaint_id", "issue", "consumer_duty_outcome", "vulnerable", "vulnerability_reason", "severity", "confidence"]]
tri = tri.rename(columns={"issue": "ai_issue"})

DB = "data/clean/complaints.db"
with sqlite3.connect(DB) as con:
    allc.to_sql("complaints", con, if_exists="replace", index=False)
    tri.to_sql("ai_triage", con, if_exists="replace", index=False)
print(f"complaints table: {len(allc):,} rows | ai_triage table: {len(tri):,} rows")

complaints table: 102,402 rows | ai_triage table: 1,000 rows


In [3]:
# 3. The agent's ONLY tool: a read-only SQL runner with guardrails
TOOL_LOG = []   # every call is recorded: the audit trail

BLOCKED = re.compile(r"\b(insert|update|delete|drop|alter|create|replace|attach|pragma|vacuum)\b", re.I)

def run_sql(query: str) -> str:
    """Run ONE read-only SQLite SELECT query on the complaints database and return the result as text.
    Tables:
      complaints(complaint_id, issue, sub_issue, company, state, tags, company_response, timely_response, words, year, month)
      ai_triage(complaint_id, ai_issue, consumer_duty_outcome, vulnerable, vulnerability_reason, severity, confidence)
    ai_triage covers only the 1,000 test complaints; join on complaint_id. vulnerable is 1/0. month is 'YYYY-MM'.
    Results are capped at 50 rows."""
    q = query.strip().rstrip(";")
    if not q.lower().startswith(("select", "with")) or BLOCKED.search(q) or ";" in q:
        out = "ERROR: only a single read-only SELECT query is allowed."
    else:
        try:
            with sqlite3.connect(f"file:{DB}?mode=ro", uri=True) as con:   # opened READ-ONLY
                res = pd.read_sql_query(q, con)
            out = res.head(50).to_string(index=False) + (f"\n... ({len(res)} rows, first 50 shown)" if len(res) > 50 else "")
        except Exception as e:
            out = f"ERROR: {e}"
    TOOL_LOG.append({"query": q, "error": out.startswith("ERROR")})
    return out

# test the guardrails yourself
print(run_sql("SELECT COUNT(*) AS n FROM complaints"))
print(run_sql("DROP TABLE complaints"))

     n
102402
ERROR: only a single read-only SELECT query is allowed.


In [4]:
# 4. The agent: Gemini + the run_sql tool. The SDK runs the loop automatically
#    (model asks for SQL -> we run it -> result goes back -> model continues), up to 8 tool calls.
AGENT_SYSTEM = """You are a careful data analyst for a credit card complaints team.
Answer the question by querying the database with the run_sql tool. Never guess numbers - always query.
If a query returns an ERROR, read it, fix the SQL and try again.
Percentages: give one decimal place. Company / issue / state names: copy them exactly as they appear in the data.
End your reply with one line in the form:  FINAL ANSWER: <answer>"""

def ask_agent(question, tries=4):
    for attempt in range(tries):
        try:
            start = len(TOOL_LOG)
            resp = client.models.generate_content(
                model=MODEL, contents=question,
                config=types.GenerateContentConfig(
                    system_instruction=AGENT_SYSTEM, tools=[run_sql], temperature=0,
                    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=8)))
            calls = TOOL_LOG[start:]
            text = resp.text or ""
            m = re.search(r"FINAL ANSWER:\s*(.+)", text)
            return {"answer": m.group(1).strip() if m else text.strip()[-200:],
                    "tool_calls": len(calls), "sql_errors_fixed": sum(c["error"] for c in calls),
                    "last_sql": calls[-1]["query"] if calls else ""}
        except Exception as e:
            wait = 65 if "429" in str(e) else 20 * (attempt + 1)
            print(f"  {type(e).__name__}: {str(e)[:90]} -> retrying in {wait}s")
            time.sleep(wait)
    return {"answer": "FAILED", "tool_calls": 0, "sql_errors_fixed": 0, "last_sql": ""}

r = ask_agent("How many complaints are in the complaints table?")
print(r)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'answer': '102,402', 'tool_calls': 1, 'sql_errors_fixed': 0, 'last_sql': 'SELECT count(*) FROM complaints'}


In [5]:
# 5. The exam: 10 business questions. The correct answers come from OUR OWN SQL (the answer key).
EXAM = [
 ("How many complaints are in the complaints table?",
  "SELECT COUNT(*) FROM complaints", "number"),
 ("Which company has the most complaints?",
  "SELECT company FROM complaints GROUP BY company ORDER BY COUNT(*) DESC LIMIT 1", "text"),
 ("How many complaints with the issue 'Fees or interest' were received in 2025?",
  "SELECT COUNT(*) FROM complaints WHERE issue='Fees or interest' AND year=2025", "number"),
 ("What percentage of all complaints were 'Closed with monetary relief'?",
  "SELECT ROUND(100.0*SUM(company_response='Closed with monetary relief')/COUNT(*),1) FROM complaints", "number"),
 ("Which state has the most complaints?",
  "SELECT state FROM complaints WHERE state IS NOT NULL GROUP BY state ORDER BY COUNT(*) DESC LIMIT 1", "text"),
 ("Which issue had the largest increase in number of complaints from 2024 to 2025?",
  "SELECT issue FROM complaints GROUP BY issue ORDER BY SUM(year=2025)-SUM(year=2024) DESC LIMIT 1", "text"),
 ("How many complaints have a tag that includes 'Older American'?",
  "SELECT COUNT(*) FROM complaints WHERE tags LIKE '%Older American%'", "number"),
 ("Among the complaints triaged by the AI, what percentage were flagged as vulnerable?",
  "SELECT ROUND(100.0*AVG(vulnerable),1) FROM ai_triage", "number"),
 ("Among AI-triaged complaints flagged as vulnerable, which original issue (from the complaints table) is most common?",
  "SELECT c.issue FROM ai_triage a JOIN complaints c USING(complaint_id) WHERE a.vulnerable=1 GROUP BY c.issue ORDER BY COUNT(*) DESC LIMIT 1", "text"),
 ("Among companies with at least 1,000 complaints, which has the highest percentage of untimely responses (timely_response = 'No')?",
  "SELECT company FROM complaints GROUP BY company HAVING COUNT(*)>=1000 ORDER BY AVG(timely_response='No') DESC LIMIT 1", "text"),
]

with sqlite3.connect(DB) as con:
    key = [str(con.execute(sql).fetchone()[0]) for _, sql, _ in EXAM]
for (q, _, _), k in zip(EXAM, key):
    print(f"{k:>40}  <- {q[:70]}")

                                  102402  <- How many complaints are in the complaints table?
       CAPITAL ONE FINANCIAL CORPORATION  <- Which company has the most complaints?
                                    4810  <- How many complaints with the issue 'Fees or interest' were received in
                                    16.6  <- What percentage of all complaints were 'Closed with monetary relief'?
                                      CA  <- Which state has the most complaints?
Problem with a purchase shown on your statement  <- Which issue had the largest increase in number of complaints from 2024
                                   10780  <- How many complaints have a tag that includes 'Older American'?
                                    16.7  <- Among the complaints triaged by the AI, what percentage were flagged a
Problem with a purchase shown on your statement  <- Among AI-triaged complaints flagged as vulnerable, which original issu
                     SYNCHRONY FINANCIA

In [6]:
# 6. Run the exam (about 10 x 2-4 AI calls, a few minutes)
def is_correct(answer, truth, kind):
    a = str(answer).lower().replace(",", "").replace("%", "")
    if kind == "number":
        nums = [float(x) for x in re.findall(r"-?\d+\.?\d*", a)]
        return any(abs(n - float(truth)) <= max(0.1, 0.005 * abs(float(truth))) for n in nums)
    return str(truth).lower() in a

rows = []
for (q, _, kind), truth in zip(EXAM, key):
    r = ask_agent(q)
    rows.append({"question": q, "correct_answer": truth, "agent_answer": r["answer"],
                 "correct": is_correct(r["answer"], truth, kind),
                 "tool_calls": r["tool_calls"], "sql_errors_self_fixed": r["sql_errors_fixed"], "agent_sql": r["last_sql"]})
    print("OK " if rows[-1]["correct"] else "X  ", q[:80])
    time.sleep(5)

exam = pd.DataFrame(rows)
exam.to_csv("outputs/agent_exam.csv", index=False)
print(f"\nAgent score: {exam['correct'].sum()}/{len(exam)} | avg tool calls per question: {exam['tool_calls'].mean():.1f} "
      f"| SQL errors it fixed itself: {exam['sql_errors_self_fixed'].sum()}")
exam[["question", "correct_answer", "agent_answer", "correct", "tool_calls"]]

OK  How many complaints are in the complaints table?
OK  Which company has the most complaints?
OK  How many complaints with the issue 'Fees or interest' were received in 2025?
OK  What percentage of all complaints were 'Closed with monetary relief'?
OK  Which state has the most complaints?
OK  Which issue had the largest increase in number of complaints from 2024 to 2025?
OK  How many complaints have a tag that includes 'Older American'?
OK  Among the complaints triaged by the AI, what percentage were flagged as vulnerab
OK  Among AI-triaged complaints flagged as vulnerable, which original issue (from th
OK  Among companies with at least 1,000 complaints, which has the highest percentage

Agent score: 10/10 | avg tool calls per question: 1.6 | SQL errors it fixed itself: 0


,question,correct_answer,agent_answer,correct,tool_calls
0,How many complaints are in the complaints table?,102402,"102,402",True,1
1,Which company has the most complaints?,CAPITAL ONE FINANCIAL CORPORATION,CAPITAL ONE FINANCIAL CORPORATION,True,1
2,How many complaints with the issue 'Fees or in...,4810,4810,True,1
3,What percentage of all complaints were 'Closed...,16.6,16.6%,True,2
4,Which state has the most complaints?,CA,CA,True,3
5,Which issue had the largest increase in number...,Problem with a purchase shown on your statement,Problem with a purchase shown on your statement,True,2
6,How many complaints have a tag that includes '...,10780,"10,780",True,2
7,"Among the complaints triaged by the AI, what p...",16.7,16.7%,True,1
8,Among AI-triaged complaints flagged as vulnera...,Problem with a purchase shown on your statement,Problem with a purchase shown on your statement,True,2
9,"Among companies with at least 1,000 complaints...",SYNCHRONY FINANCIAL,SYNCHRONY FINANCIAL,True,1


In [7]:
# 7. Look inside the agent's work: the SQL it wrote for each question (the audit trail)
for _, r in exam.iterrows():
    print(("OK " if r["correct"] else "X  ") + r["question"])
    print("   SQL:", r["agent_sql"][:300], "\n")

OK How many complaints are in the complaints table?
   SQL: SELECT count(*) FROM complaints 

OK Which company has the most complaints?
   SQL: SELECT company, COUNT(*) as cnt FROM complaints GROUP BY company ORDER BY cnt DESC LIMIT 1 

OK How many complaints with the issue 'Fees or interest' were received in 2025?
   SQL: SELECT count(*) FROM complaints WHERE issue = 'Fees or interest' AND year = 2025 

OK What percentage of all complaints were 'Closed with monetary relief'?
   SQL: SELECT company_response, COUNT(*) as cnt, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints), 1) as pct FROM complaints GROUP BY company_response 

OK Which state has the most complaints?
   SQL: SELECT state, COUNT(*) as cnt FROM complaints GROUP BY state ORDER BY cnt DESC LIMIT 1 

OK Which issue had the largest increase in number of complaints from 2024 to 2025?
   SQL: SELECT 
  c.issue,
  SUM(CASE WHEN c.year = 2024 THEN 1 ELSE 0 END) AS cnt_2024,
  SUM(CASE WHEN c.year = 2025 THEN 1 ELSE 0 EN